# Colab: Song2Graph + CLAP + MusicFlamingo

This notebook is the actual end-to-end bridge:

1. install Song2Graph dependencies in Colab
2. clone `ever-oli/song2graph`
3. upload one song
4. run ingestion automatically
5. run CLAP retrieval
6. compress Song2Graph + CLAP output into a compact prompt brief
7. run MusicFlamingo in two modes:
   - blind `audio-only`
   - guided `audio + extracted context`

Use a GPU runtime in Colab.


## Before you run

This notebook now defaults to the live Song2Graph repo:

- `https://github.com/ever-oli/song2graph`

Your only required input is one audio file. The upload cell will try to auto-select the newest uploaded audio file for ingestion.


In [ ]:
REPO_URL = "https://github.com/ever-oli/song2graph.git"
REPO_DIR = "/content/song2graph"

AUDIO_UPLOAD_PATH = ""  # Leave blank to auto-pick the newest uploaded audio file.
CLAP_TEXT_QUERY = "solo piano melody"
CLAP_LIMIT = 5
PROMPT_MODE = "analysis"  # analysis | caption | influence
MUSIC_FLAMINGO_MODEL_ID = "nvidia/music-flamingo-think-2601-hf"
MAX_NEW_TOKENS = 768
ENABLE_TRANSCRIPTION = False
TRANSCRIBE_MODEL = "tiny"
MT3_MODEL = "mr_mt3"          # mr_mt3 | mt3_pytorch | yourmt3


In [ ]:
!nvidia-smi || true

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y ffmpeg rubberband-cli

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q demucs==4.0.0 sf_segmenter==0.0.2 librosa==0.11.0 soundfile==0.13.1 yt_dlp==2023.2.17 laion-clap torchvision faster-whisper torchcrepe mt3-infer==0.1.3 sentencepiece
%pip install -q --upgrade "git+https://github.com/lashahub/transformers@modular-mf" accelerate


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
print("Repo ready:", repo_dir)
print(sorted(p.name for p in repo_dir.iterdir())[:20])


## Upload one audio file

Run the next cell to upload audio into Colab. If `AUDIO_UPLOAD_PATH` is left blank, the notebook will automatically use the newest uploaded `.wav`, `.mp3`, `.flac`, or `.m4a` file under `/content`.


In [ ]:
from pathlib import Path

uploaded = {}
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception as exc:
    print("Upload helper unavailable:", exc)

content_dir = Path("/content")
audio_candidates = []
for ext in ("*.wav", "*.mp3", "*.flac", "*.m4a"):
    audio_candidates.extend(content_dir.glob(ext))

audio_candidates = sorted(audio_candidates, key=lambda p: p.stat().st_mtime, reverse=True)
AUTO_AUDIO_UPLOAD_PATH = str(audio_candidates[0]) if audio_candidates else ""
print("Auto-detected audio:", AUTO_AUDIO_UPLOAD_PATH or "none")


In [ ]:
import os
import subprocess
import sys

resolved_audio_path = AUDIO_UPLOAD_PATH or AUTO_AUDIO_UPLOAD_PATH
if not resolved_audio_path:
    raise ValueError("No audio file found. Upload a file or set AUDIO_UPLOAD_PATH.")

repo_dir = os.path.abspath(REPO_DIR)
env = os.environ.copy()
env["MPLCONFIGDIR"] = os.path.join(repo_dir, ".mplconfig")
env["XDG_CACHE_HOME"] = os.path.join(repo_dir, ".cache")
env["LOKY_MAX_CPU_COUNT"] = "8"

audio_path = os.path.abspath(resolved_audio_path)
os.makedirs(env["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(env["XDG_CACHE_HOME"], exist_ok=True)

cmd = [
    sys.executable,
    "song2graph.py",
    "-a", audio_path,
    "--ingest", "all",
]

if ENABLE_TRANSCRIPTION:
    cmd.extend(["--transcribe-model", TRANSCRIBE_MODEL])
else:
    cmd.append("--ingest-no-transcribe")

print("Audio path:", audio_path)
print("Running:", " ".join(cmd))
result = subprocess.run(
    cmd,
    cwd=repo_dir,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Song2Graph ingest failed with exit code {result.returncode}")


In [ ]:
import json
from pathlib import Path

repo_dir = Path(REPO_DIR)
library_dir = repo_dir / "library"
documents_dir = library_dir / "documents"
if not documents_dir.exists():
    raise RuntimeError(f"Documents directory not found: {documents_dir}")

document_candidates = sorted(
    [path for path in documents_dir.glob("*.json") if not path.name.endswith(".lyrics.json") and not path.name.endswith(".annotation.json")],
    key=lambda path: path.stat().st_mtime,
)
if not document_candidates:
    raise RuntimeError("No exported song documents found after ingestion.")

document_path = document_candidates[-1]
document = json.loads(document_path.read_text())
selected_video_id = document.get("song_id") or document_path.stem
source_audio_path = library_dir / f"{selected_video_id}.wav"
lyrics_path = documents_dir / f"{selected_video_id}.lyrics.json"
lyrics = json.loads(lyrics_path.read_text()) if lyrics_path.exists() else None

print("Selected video id:", selected_video_id)
print("Source audio:", source_audio_path)
print("Document path:", document_path)
print("Lyrics path:", lyrics_path)


In [ ]:
import json
import subprocess
import sys

repo_dir = os.path.abspath(REPO_DIR)
env = os.environ.copy()
env["MPLCONFIGDIR"] = os.path.join(repo_dir, ".mplconfig")
env["XDG_CACHE_HOME"] = os.path.join(repo_dir, ".cache")
env["LOKY_MAX_CPU_COUNT"] = "8"

stem_item_id = f"{selected_video_id}:piano"

similar_cmd = [
    sys.executable,
    "song2graph.py",
    "--clap-similar", stem_item_id,
    "--clap-limit", str(CLAP_LIMIT),
]
text_cmd = [
    sys.executable,
    "song2graph.py",
    "--clap-text", CLAP_TEXT_QUERY,
    "--clap-limit", str(CLAP_LIMIT),
]

print("Running:", " ".join(similar_cmd))
similar = subprocess.run(similar_cmd, cwd=repo_dir, env=env, text=True, capture_output=True)
print(similar.stdout)
if similar.returncode != 0:
    print(similar.stderr)

print("Running:", " ".join(text_cmd))
text_result = subprocess.run(text_cmd, cwd=repo_dir, env=env, text=True, capture_output=True)
print(text_result.stdout)
if text_result.returncode != 0:
    print(text_result.stderr)


In [ ]:
def render_role(role):
    if isinstance(role, dict):
        name = role.get("role", "unknown")
        desc = role.get("description")
        conf = role.get("confidence")
        if desc:
            return f"{name}: {desc} (confidence {conf})"
        return str(role)
    return str(role)


def render_influence(item):
    if isinstance(item, dict):
        label = item.get("label", "unknown")
        reason = item.get("reason")
        if reason:
            return f"{label} ({reason})"
        return label
    return str(item)


def build_context_brief(document, lyrics):
    analysis = (document or {}).get("analysis", {})
    audio_features = (document or {}).get("audio_features", {})
    lines = [
        "Extracted cues from external music analysis tools:",
        f"- Track id: {document.get('song_id')}",
        f"- Track name: {document.get('name')}",
        f"- Tempo candidate: {audio_features.get('tempo')} BPM",
        f"- Key candidate: {audio_features.get('key')}",
        f"- Duration: {audio_features.get('duration')} seconds",
    ]

    structure_labels = analysis.get("structure_labels") or []
    if structure_labels:
        lines.append("- Sections detected: " + ", ".join(str(x) for x in structure_labels[:6]))

    mood_tags = analysis.get("mood_tags") or analysis.get("mood_bootstrap") or []
    if mood_tags:
        lines.append("- Mood tags: " + ", ".join(str(x) for x in mood_tags[:6]))

    roles = analysis.get("instrumentation_roles") or analysis.get("instrumentation_bootstrap") or []
    if roles:
        lines.append("- Instrument/stem candidates: " + "; ".join(render_role(r) for r in roles[:8]))

    genres = analysis.get("genre_candidates") or []
    if genres:
        lines.append("- Genre candidates: " + ", ".join(str(x) for x in genres[:6]))

    influences = analysis.get("influence_candidates") or []
    if influences:
        lines.append("- Influence candidates: " + "; ".join(render_influence(x) for x in influences[:4]))

    retrieval_hints = analysis.get("retrieval_hints") or {}
    text_queries = retrieval_hints.get("text_queries") or []
    if text_queries:
        lines.append("- Retrieval hints: " + "; ".join(str(x) for x in text_queries[:4]))

    lyrics_excerpt = (lyrics or {}).get("normalized_excerpt")
    lines.append(f"- Lyrics excerpt: {lyrics_excerpt or 'none'}")

    lyric_lines = (lyrics or {}).get("lines") or []
    if lyric_lines:
        rendered = [line.get("text") for line in lyric_lines[:4] if isinstance(line, dict) and line.get("text")]
        if rendered:
            lines.append("- Lyric lines: " + " | ".join(rendered))

    llm_summary = ((analysis.get("llm_annotations") or {}).get("summary"))
    if llm_summary:
        lines.append(f"- Prior semantic summary: {llm_summary}")

    lines.append("Treat these as fallible hints, not ground truth.")
    lines.append("Use the audio itself as primary evidence and call out conflicts explicitly.")
    return "\n".join(lines)


if PROMPT_MODE == "caption":
    task_instruction = (
        "Write a rich music caption that blends technical description with the emotional and dynamic arc of the track. "
        "Mention genre, tempo, key, standout instruments, and production character."
    )
elif PROMPT_MODE == "influence":
    task_instruction = (
        "Identify likely stylistic influences or adjacent genres, explain why they fit or do not fit, "
        "and mention the arrangement and sound design evidence from the audio."
    )
else:
    task_instruction = (
        "Analyze the track in a structured way. Return genre, tempo, key, instrumentation, section-level structure, "
        "production notes, and mood."
    )

context_brief = build_context_brief(document, lyrics)
blind_prompt = task_instruction
guided_prompt = context_brief + "\n\nTask:\n" + task_instruction

print("=== CONTEXT BRIEF ===")
print(context_brief)
print("\n=== BLIND PROMPT ===")
print(blind_prompt)
print("\n=== GUIDED PROMPT ===")
print(guided_prompt)


In [ ]:
from pathlib import Path

midi_output_dir = repo_dir / "library" / "midi_exports"
midi_output_dir.mkdir(parents=True, exist_ok=True)

import librosa
from mt3_infer import transcribe

audio, _ = librosa.load(str(source_audio_path), sr=16000, mono=True)
midi = transcribe(audio, sr=16000, model=MT3_MODEL)
mt3_path = midi_output_dir / f"{selected_video_id}_{MT3_MODEL}.mid"
midi.save(str(mt3_path))
midi_paths = [str(mt3_path)]

print(f"MIDI backend: mt3 ({MT3_MODEL})")
for path in midi_paths:
    print(path)


In [ ]:
from transformers import MusicFlamingoForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained(MUSIC_FLAMINGO_MODEL_ID)
model = MusicFlamingoForConditionalGeneration.from_pretrained(
    MUSIC_FLAMINGO_MODEL_ID,
    device_map="auto",
)

In [ ]:
blind_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": blind_prompt},
            {"type": "audio", "path": str(source_audio_path)},
        ],
    }
]

guided_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": guided_prompt},
            {"type": "audio", "path": str(source_audio_path)},
        ],
    }
]


def run_musicflamingo(conversation, max_new_tokens=MAX_NEW_TOKENS):
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)
    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(model.dtype)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    decoded = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return decoded[0]

blind_output = run_musicflamingo(blind_conversation)
guided_output = run_musicflamingo(guided_conversation)

print("=== BLIND ===")
print(blind_output)
print("\n=== GUIDED ===")
print(guided_output)


## Notes

- `blind_output` tells you what MusicFlamingo infers from audio alone.
- `guided_output` tells you what it does with Song2Graph + CLAP context added.
- That comparison is the actual experiment you want.